In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine, text


# database connections 


In [ ]:
with open('../../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['OLTP']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


# Extract


In [ ]:
dim_mensajero = pd.read_sql_query("""
SELECT m.id AS id_mensajero, m.user_id, m.activo, m.telefono,
       m.ciudad_operacion_id AS id_ciudad_operacion, u.username,
       TRIM(CONCAT(COALESCE(u.first_name, ''), ' ', COALESCE(u.last_name, ''))) AS nombre,
       c.nombre AS ciudad_operacion
FROM clientes_mensajeroaquitoy m
LEFT JOIN auth_user u ON u.id = m.user_id
LEFT JOIN ciudad c ON c.ciudad_id = m.ciudad_operacion_id
""", co_sa)
vehiculo = pd.read_sql_query("""
SELECT s.mensajero_id AS id_mensajero, tv.nombre AS tipo_vehiculo, COUNT(*) AS n
FROM mensajeria_servicio s
JOIN mensajeria_tipovehiculo tv ON tv.id = s.tipo_vehiculo_id
WHERE s.mensajero_id IS NOT NULL
GROUP BY s.mensajero_id, tv.nombre
""", co_sa)
dim_mensajero.head()


# Transformations


In [ ]:
vehiculo = vehiculo.sort_values(['id_mensajero', 'n'], ascending=[True, False])
vehiculo = vehiculo.drop_duplicates(subset=['id_mensajero'], keep='first')
dim_mensajero = dim_mensajero.merge(vehiculo[['id_mensajero', 'tipo_vehiculo']], on='id_mensajero', how='left')
dim_mensajero['nombre'] = dim_mensajero['nombre'].replace({'': np.nan, ' ': np.nan})
dim_mensajero['nombre'] = dim_mensajero['nombre'].fillna(dim_mensajero['username'])
dim_mensajero.replace({np.nan: 'NO APLICA', ' ': 'NO APLICA', '': 'NO APLICA'}, inplace=True)
dim_mensajero['activo'] = dim_mensajero['activo'].replace({'NO APLICA': False}).fillna(False)
dim_mensajero['id_ciudad_operacion'] = pd.to_numeric(dim_mensajero['id_ciudad_operacion'], errors='coerce').fillna(-1).astype('Int64')
dim_mensajero.drop(columns=['user_id'], inplace=True, errors='ignore')

unknown = pd.DataFrame([{
    'id_mensajero': -1, 'username': 'NO APLICA', 'nombre': 'NO APLICA', 'telefono': 'NO APLICA',
    'activo': False, 'id_ciudad_operacion': -1, 'ciudad_operacion': 'NO APLICA', 'tipo_vehiculo': 'NO APLICA',
}])
dim_mensajero = pd.concat([unknown, dim_mensajero], ignore_index=True)
dim_mensajero["saved"] = date.today()
dim_mensajero.head()


# load


In [ ]:
with etl_conn.begin() as conn:
    conn.execute(text('Delete from dim_mensajero'))
dim_mensajero.to_sql('dim_mensajero', etl_conn, if_exists='append', index=False)
